Gemini Automated Evaluation Script for GPT-4o, Typhoon and SeaLLM

In [ ]:
import google.generativeai as genai
import json
import time
import os

# ==========================================
# ⚙️ Step 1: API and Model Configuration
# ==========================================
# Read the API key from Colab Secrets (never hard-code keys in the notebook)
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Judge model: Gemini 3.6 Flash with temperature 0.0 for strict, deterministic evaluation
judge_model = genai.GenerativeModel(
    'gemini-3.6-flash',
    generation_config=genai.GenerationConfig(temperature=0.0)
)

# ⚠️ Ensure these keys match your JSON structure
QUERY_KEY = "prompt_th"
GROUND_TRUTH_KEY = "ground_truth_th"

# ==========================================
# 🗂️ Step 2: File List and Model-Specific Answer Keys
# ==========================================
# "ans_key" tells the system which exact key to extract the answer from for each file
files_to_process = [
    # --- GPT-4o ---
    {"in": "history_gpt4o_answers_COMPLETE.json", "out": "evaluated_gpt4o_history.json", "ans_key": "gpt4o_answer"},
    {"in": "law_gpt4o_answers.json", "out": "evaluated_gpt4o_law.json", "ans_key": "gpt4o_answer"},
    {"in": "culture_gpt4o_answers.json", "out": "evaluated_gpt4o_culture.json", "ans_key": "gpt4o_answer"},

    # --- Typhoon ---
    {"in": "history_typhoon_answers.json", "out": "evaluated_typhoon_history.json", "ans_key": "typhoon_answer"},
    {"in": "law_typhoon_answers.json", "out": "evaluated_typhoon_law.json", "ans_key": "typhoon_answer"},
    {"in": "culture_typhoon_answers.json", "out": "evaluated_typhoon_culture.json", "ans_key": "typhoon_answer"},

    # --- SeaLLM ---
    {"in": "history_seallm_answers_LOCAL.json", "out": "evaluated_seallm_history.json", "ans_key": "seallm_answer"},
    {"in": "law_seallm_answers_LOCAL.json", "out": "evaluated_seallm_law.json", "ans_key": "seallm_answer"},
    {"in": "culture_seallm_answers_LOCAL.json", "out": "evaluated_seallm_culture.json", "ans_key": "seallm_answer"}
]

# ==========================================
# 📝 Step 3: Judge Prompt Function
# ==========================================
def get_evaluation(query, ground_truth, model_answer):
    prompt = f"""[System Role]
You are an impartial and rigorous factual evaluator. You must evaluate the model's generation SOLELY based on the facts provided in the [Reference Ground Truth]. You are strictly prohibited from utilizing your own internal knowledge base or external contextual memory to verify the response.

[Target Query]
{query}

[Reference Ground Truth]
{ground_truth}

[Model Generation]
{model_answer}

[Evaluation Rubric]
Classify the [Model Generation] into EXACTLY ONE of these four categories:
1. Factually Accurate: Correctly identifies core facts from the ground truth without contradicting information.
2. Intrinsic Hallucination: Directly contradicts specific facts, dates, or clauses provided in the ground truth.
3. Extrinsic Hallucination: Introduces highly plausible but unverified/absent entities or events NOT found in the ground truth.
4. Cultural Context Confusion: Applies Western sociological, historical, or legal principles to the localized Thai query incorrectly.

[Output Format]
You must think step-by-step (Chain-of-Thought). Output strictly in this format:
Reasoning: <your step-by-step logic comparing generation to ground truth>
Category: <Exact name of the category from the rubric>
"""

    try:
        response = judge_model.generate_content(
            prompt,
            request_options={"timeout": 30}
        )
        text = response.text
        reasoning = text.split("Category:")[0].replace("Reasoning:", "").strip()
        category = text.split("Category:")[1].strip()
        return reasoning, category

    except Exception as e:
        print(f"⚠️ API Error: {e}")
        return "Error", "Unknown/Error"

# ==========================================
# 🚀 Step 4: Execute Batch Evaluation Loop
# ==========================================
print("🚀 Starting the automated evaluation process...\n")

for file_info in files_to_process:
    input_file = file_info["in"]
    output_file = file_info["out"]
    answer_key = file_info["ans_key"]

    # --- Loading Progress ---
    if os.path.exists(output_file):
        print(f"\n📂 Loading existing progress: {output_file}")
        with open(output_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
    elif os.path.exists(input_file):
        print(f"\n📂 Starting fresh file: {input_file}")
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        print(f"⏭️ Skipping: File '{input_file}' not found.")
        continue

    # --- Processing ---
    for i, row in enumerate(data):
        # Skip if already evaluated
        if "gemini_category" in row and row["gemini_category"] not in ["", "Unknown/Error", None]:
            continue

        query = row.get(QUERY_KEY, "")
        ground_truth = row.get(GROUND_TRUTH_KEY, "")
        model_answer = row.get(answer_key, "")

        print(f"   ⏳ Evaluating item {i+1}/{len(data)}...", end=" ", flush=True)

        reasoning, category = get_evaluation(query, ground_truth, model_answer)

        row['gemini_reasoning'] = reasoning
        row['gemini_category'] = category

        print(f"Done! -> [{category}]")

        # Save every single row to prevent data loss
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        time.sleep(3) # Prevent API Rate limits

    print(f"✅ Evaluation complete for {input_file}!\n" + "-"*40)

print("\n🎉 Batch evaluation completed successfully!")

🚀 Starting the automated evaluation process...


📂 Loading existing progress: evaluated_gpt4o_history.json
✅ Evaluation complete for history_gpt4o_answers_COMPLETE.json!
----------------------------------------

📂 Loading existing progress: evaluated_gpt4o_law.json
✅ Evaluation complete for law_gpt4o_answers.json!
----------------------------------------

📂 Loading existing progress: evaluated_gpt4o_culture.json
✅ Evaluation complete for culture_gpt4o_answers.json!
----------------------------------------

📂 Loading existing progress: evaluated_typhoon_history.json
✅ Evaluation complete for history_typhoon_answers.json!
----------------------------------------

📂 Loading existing progress: evaluated_typhoon_law.json
✅ Evaluation complete for law_typhoon_answers.json!
----------------------------------------

📂 Loading existing progress: evaluated_typhoon_culture.json
✅ Evaluation complete for culture_typhoon_answers.json!
----------------------------------------

📂 Loading existing pro

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Done! -> [Extrinsic Hallucination]
   ⏳ Evaluating item 34/50... Done! -> [Intrinsic Hallucination]
   ⏳ Evaluating item 36/50... Done! -> [Extrinsic Hallucination]
✅ Evaluation complete for culture_seallm_answers_LOCAL.json!
----------------------------------------

🎉 Batch evaluation completed successfully!


Summarize data

In [ ]:
import json
import pandas as pd
import os

# ==========================================
# 📊 Step 1: Define Evaluated Files and Categories
# ==========================================
# List of the 9 evaluated files, specifying Model and Domain
evaluated_files = [
    # GPT-4o
    {"model": "GPT-4o", "domain": "History", "file": "evaluated_gpt4o_history.json"},
    {"model": "GPT-4o", "domain": "Law", "file": "evaluated_gpt4o_law.json"},
    {"model": "GPT-4o", "domain": "Culture", "file": "evaluated_gpt4o_culture.json"},
    # Typhoon
    {"model": "Typhoon", "domain": "History", "file": "evaluated_typhoon_history.json"},
    {"model": "Typhoon", "domain": "Law", "file": "evaluated_typhoon_law.json"},
    {"model": "Typhoon", "domain": "Culture", "file": "evaluated_typhoon_culture.json"},
    # SeaLLM
    {"model": "SeaLLM", "domain": "History", "file": "evaluated_seallm_history.json"},
    {"model": "SeaLLM", "domain": "Law", "file": "evaluated_seallm_law.json"},
    {"model": "SeaLLM", "domain": "Culture", "file": "evaluated_seallm_culture.json"}
]

# All categories based on the defined Rubric
categories = [
    "Factually Accurate",
    "Intrinsic Hallucination",
    "Extrinsic Hallucination",
    "Cultural Context Confusion",
    "Unknown/Error" # Tracked to report excluded items in Methodology
]

summary_data = []

print("📂 Compiling statistical data...")

# ==========================================
# 🔄 Step 2: Loop Through Files and Aggregate Data
# ==========================================
for item in evaluated_files:
    filepath = item["file"]
    model_name = item["model"]
    domain_name = item["domain"]

    # Initialize category counters for the current file
    counts = {cat: 0 for cat in categories}
    total_items = 0

    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            total_items = len(data)

            for row in data:
                cat = row.get("gemini_category", "Unknown/Error")
                # Ensure the category matches the rubric (handles slight formatting variations)
                matched_cat = next((c for c in categories if c.lower() in str(cat).lower()), "Unknown/Error")
                counts[matched_cat] += 1

        print(f"✅ Successfully read {filepath} ({total_items} items)")
    else:
        print(f"⚠️ File not found: {filepath}")

    # Format the aggregated data into a dictionary for the DataFrame
    row_data = {
        "Model": model_name,
        "Domain": domain_name,
        "Total Prompts": total_items,
        **counts
    }
    summary_data.append(row_data)

# ==========================================
# 📈 Step 3: Data Processing and Excel Export
# ==========================================
# Convert the aggregated list to a Pandas DataFrame
df = pd.DataFrame(summary_data)

# Calculate Accuracy Rate (%)
# Excluding 'Unknown/Error' from the base denominator to ensure statistical validity
df["Valid Answers"] = df["Total Prompts"] - df["Unknown/Error"]
df["Accuracy Rate (%)"] = (df["Factually Accurate"] / df["Valid Answers"] * 100).round(2)

# Save the DataFrame to an Excel file
excel_filename = "LLM_Hallucination_Results_Chapter4.xlsx"
df.to_excel(excel_filename, index=False)

print("-" * 40)
print(f"🎉 Done! Excel file saved successfully as: {excel_filename}")
display(df) # Preview the generated table in Colab

📂 Compiling statistical data...
✅ Successfully read evaluated_gpt4o_history.json (50 items)
✅ Successfully read evaluated_gpt4o_law.json (50 items)
✅ Successfully read evaluated_gpt4o_culture.json (50 items)
✅ Successfully read evaluated_typhoon_history.json (50 items)
✅ Successfully read evaluated_typhoon_law.json (50 items)
✅ Successfully read evaluated_typhoon_culture.json (50 items)
✅ Successfully read evaluated_seallm_history.json (50 items)
✅ Successfully read evaluated_seallm_law.json (50 items)
✅ Successfully read evaluated_seallm_culture.json (50 items)
----------------------------------------
🎉 Done! Excel file saved successfully as: LLM_Hallucination_Results_Chapter4.xlsx


,Model,Domain,Total Prompts,Factually Accurate,Intrinsic Hallucination,Extrinsic Hallucination,Cultural Context Confusion,Unknown/Error,Valid Answers,Accuracy Rate (%)
0,GPT-4o,History,50,8,18,24,0,0,50,16.0
1,GPT-4o,Law,50,20,14,16,0,0,50,40.0
2,GPT-4o,Culture,50,15,5,30,0,0,50,30.0
3,Typhoon,History,50,0,47,3,0,0,50,0.0
4,Typhoon,Law,50,0,34,16,0,0,50,0.0
5,Typhoon,Culture,50,1,8,41,0,0,50,2.0
6,SeaLLM,History,50,1,44,5,0,0,50,2.0
7,SeaLLM,Law,50,6,31,13,0,0,50,12.0
8,SeaLLM,Culture,50,2,20,28,0,0,50,4.0
